# Sesión 4 — MBD

## Regresión logística: del riesgo a la decisión

**Pregunta:** ¿qué probabilidad tiene un crédito de pagarse según las características de quien lo solicita y las condiciones del préstamo? La respuesta es **`y = yes`: el crédito se paga**; codificamos ese suceso como 1 y `no` como 0. La regresión logística produce probabilidades entre 0 y 1; para convertirlas en decisiones hay que elegir un umbral, que discutiremos al final.

**Puente con la presentación (16 diapositivas):** motivación, logit y modelo (3–7) → preparación y ajuste; odds ratios (8–10) → interpretación; selección (11) → AIC; ROC y AUC (12–14) → evaluación; Hosmer–Lemeshow (15) → diagnóstico. No son dos temas separados: usamos los mismos conceptos primero en fórmulas y después en datos.

El recorrido es **describir → codificar → ajustar → seleccionar → diagnosticar → interpretar odds ratios → evaluar probabilidades y decisiones en test**. Elegimos el modelo con entrenamiento; el test se utiliza solo para evaluar resultados. Una buena separación entre clases (AUC) no implica probabilidades bien calibradas, ni un único umbral sirve para todas las decisiones.

Los microejercicios prácticos aparecen al final de los bloques principales: completa los huecos reutilizando las llamadas y operaciones ya mostradas. Los ejercicios **4.1** y **4.2** son actividades integradoras para revisar con más detenimiento, primero el modelo en entrenamiento y después su evaluación en test.

En VS Code, selecciona el kernel del entorno del proyecto y ejecuta las celdas **en orden** con **Shift + Enter**. Usa **Variables** para inspeccionar tablas y **Esquema / Outline** para navegar. Los controles interactivos permiten explorar ejemplos sin cambiar el ajuste ni la evaluación final: si no aparecen, cada sección ofrece una llamada de Python alternativa. **No ajustes el umbral mirando el test.** Para reproducir todo el análisis, usa **Reiniciar** y **Ejecutar todo**. [Guía de notebooks en VS Code](https://code.visualstudio.com/docs/datascience/jupyter-notebooks).

### Variables del conjunto de datos

| Variable | Descripción |
| --- | --- |
| `status` | Situación de la cuenta corriente |
| `duration` | Duración del crédito, en meses |
| `credit.hist` | Historial crediticio |
| `purpose` | Finalidad del crédito |
| `credit.amo` | Importe del crédito |
| `savings` | Cuenta de ahorro o bonos |
| `employed.time` | Antigüedad en el empleo actual |
| `installment` | Cuota como porcentaje de la renta disponible |
| `status.sex` | Estado civil y sexo |
| `debtors` | Otros deudores o avalistas |
| `residence.time` | Tiempo en la residencia actual |
| `property` | Propiedades |
| `age` | Edad, en años |
| `installment2` | Otros planes de pago a plazos |
| `housing` | Tipo de vivienda |
| `num.credits` | Número de créditos existentes en este banco |
| `job` | Tipo de empleo |
| `num.people` | Número de personas a cargo |
| `telephone` | Disponibilidad de teléfono |
| `foreign` | Trabajador extranjero |
| `y` | Crédito pagado: `no` o `yes` |

## Parte 1. Construir el modelo

### Preparar la sesión

La siguiente celda vacía las variables de la sesión. Después importamos las librerías necesarias.
Si falta alguna en el kernel seleccionado, descomenta y ejecuta la línea de instalación.

In [ ]:
# %pip install numpy pandas matplotlib scipy statsmodels scikit-learn threadpoolctl ipywidgets ipykernel

In [ ]:
from base64 import b64encode
from io import BytesIO
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from IPython.display import HTML as HTMLDisplay, display
from ipywidgets import Dropdown, FloatSlider, HTML, HBox, Layout, VBox
from scipy.optimize import brentq
from scipy.stats import chi2, norm
from sklearn.metrics import roc_curve, roc_auc_score
from statsmodels.graphics.mosaicplot import mosaic
from threadpoolctl import threadpool_limits

%matplotlib inline
%config InteractiveShell.ast_node_interactivity = "all"
COLOR_DATOS = "#9fc3e4"
COLOR_CLAVE = "#800000"
COLOR_BORDE = "#666666"
plt.rcParams.update({"figure.figsize": (11, 6), "figure.dpi": 110,
                     "axes.titlesize": 18, "axes.labelsize": 14,
                     "xtick.labelsize": 12, "ytick.labelsize": 12,
                     "legend.fontsize": 12})
pd.set_option("display.max_columns", 25)


def imagen_widget(fig, descripcion):
    """Renderiza una figura en HTML para que ipywidgets pueda actualizarla sin acumular salidas."""
    imagen = BytesIO()
    fig.savefig(imagen, format="png", dpi=110, bbox_inches="tight")
    plt.close(fig)
    return (f"<img src='data:image/png;base64,{b64encode(imagen.getvalue()).decode()}' "
            f"alt='{descripcion}' style='max-width: 100%; height: auto;'>")


def mostrar_html(contenido):
    """Alternativa estática cuando el editor no presenta widgets."""
    display(HTMLDisplay(contenido))

### Lectura de datos e inspección

Localizamos la carpeta `datos` del proyecto y leemos el conjunto de entrenamiento.
Las columnas de texto se representan como variables categóricas.

In [ ]:
DATA_PATH = Path.cwd().parent.parent / "datos"

datos = pd.read_csv(DATA_PATH / "bank0_train.txt", sep=";")
for columna in datos.select_dtypes(include=["object", "str"]):
    datos[columna] = datos[columna].astype("category")

In [ ]:
datos

In [ ]:
datos.describe(include="all")

### Descriptiva bivariante

Antes de ajustar un modelo, comprobamos tipos, distribución y relación de cada predictor con la respuesta. `datos.dtypes` distingue variables numéricas de categóricas; la lista `var_cat` excluye deliberadamente `y` para no usar la respuesta como predictor. En los gráficos siguientes buscamos patrones y tamaños de grupo, **no** efectos causales ni conclusiones ajustadas por otras variables.

#### Variables categóricas

In [ ]:
datos.dtypes

In [ ]:
var_cat = [columna for columna in datos.select_dtypes(include="category") if columna != "y"]
var_cat

#### Diagramas de mosaico

El ejemplo usa `status`; cambia el argumento de `mosaico_pago()` por otro nombre de `var_cat` y vuelve a ejecutar la celda para comparar variables categóricas. La **anchura** de cada bloque de `y` refleja cuántos créditos pagan o no pagan; dentro de él, la **altura** de cada categoría es su frecuencia condicional. Por tanto, los mosaicos comparan la composición de cada grupo de respuesta, no directamente $P(\mathrm{paga}\mid\mathrm{categoría})$. Consulta la tabla de frecuencias para ver los tamaños de cada combinación, sobre todo en categorías pequeñas que apenas caben en el gráfico.

**Prueba:** cambia de variable y busca una categoría con pocos casos. ¿Por qué conviene ser prudente al interpretar su rectángulo?

In [ ]:
def mosaico_pago(variable):
    import textwrap
    fig, ax = plt.subplots(figsize=(13, 7))
    _, rectangulos = mosaic(datos, ["y", variable], ax=ax,
           properties=lambda clave: {"facecolor": COLOR_CLAVE if clave[0] == "no" else COLOR_DATOS,
                                       "edgecolor": COLOR_BORDE},
           labelizer=lambda _: "",
           title=f"Composición por pago: {variable}", axes_label=False)
    for clave, (x, y_rect, ancho, alto) in rectangulos.items():
        if alto >= 0.08:
            ax.text(x + ancho / 2, y_rect + alto / 2, textwrap.fill(str(clave[1]), width=22),
                    ha="center", va="center", fontsize=10)
    centros = []
    for clase in ["no", "yes"]:
        x, y_rect, ancho, alto = next(rect for clave, rect in rectangulos.items() if clave[0] == clase)
        centros.append(x + ancho / 2)
    ax.set_xticks(centros)
    ax.set_xticklabels(["No paga", "Paga"])
    ax.set_yticks([])
    ax.set(xlabel="Pago observado", ylabel=f"Categorías de {variable}")
    fig.tight_layout()
    plt.show()
    display(pd.crosstab(datos[variable], datos["y"]))

In [ ]:
mosaico_pago("status")  # Prueba también otros nombres de var_cat.

#### Microejercicio práctico — comparar otra variable categórica

Reutiliza la llamada anterior y escoge una variable de `var_cat` distinta de `status`. Observa si las proporciones de pago parecen variar entre categorías; ten en cuenta el tamaño de cada grupo que muestra la tabla.

In [ ]:
try:
    mosaico_pago(...)  # Sustituye por el nombre de otra variable categórica que aparezca en var_cat.
except Exception as error:
    plt.close("all")  # Cierra una figura parcial si el nombre no es válido.
    print("El microejercicio no está bien implementado todavía. Revisa el nombre de la variable y vuelve a intentarlo.")
    print(f"Detalle: {type(error).__name__}: {error}")

#### Densidad condicional para variables numéricas

Aquí estimamos $P(y\mid x)$: para cada valor de la variable numérica calculamos densidades gaussianas ponderadas por la frecuencia de cada clase y normalizamos su suma. La curva se evalúa en **512 puntos** y no es todavía el modelo logístico. En zonas sin observaciones, esta estimación puede ser inestable; evita interpretar los extremos como certezas.

El ejemplo usa `age` y suavizado 1. **Explora:** edita los argumentos de `densidad_pago()` para cambiar la variable (consulta `var_num`) y el suavizado. Un ancho pequeño muestra más oscilaciones; uno grande oculta detalles. ¿Se mantiene el patrón principal con ambos? Fíjate también en que el eje vertical muestra una **probabilidad**, no el número de créditos.

In [ ]:
var_num = list(datos.select_dtypes(include="number").columns)
var_num

In [ ]:
def densidad_pago(variable, suavizado, vista_widget=False):
    x = datos[variable].to_numpy(dtype=float)
    escala = min(x.std(ddof=1), (np.quantile(x, 0.75) - np.quantile(x, 0.25)) / 1.34)
    if escala <= 0:
        escala = x.std(ddof=1) or 1.0
    ancho = 0.9 * escala * len(x) ** (-0.2) * suavizado
    rejilla = np.linspace(x.min() - 3 * ancho, x.max() + 3 * ancho, 512)
    densidades = []
    for clase in ["no", "yes"]:
        valores = x[(datos["y"] == clase).to_numpy()]
        kernels = norm.pdf((rejilla[:, None] - valores[None, :]) / ancho) / ancho
        densidades.append(kernels.sum(axis=1) / len(x))
    densidades = np.asarray(densidades)
    probabilidades = densidades / densidades.sum(axis=0)
    fig, ax = plt.subplots()
    ax.stackplot(rejilla, probabilidades, labels=["No paga", "Paga"],
                 colors=[COLOR_CLAVE, COLOR_DATOS], edgecolor=COLOR_BORDE, linewidth=0.5)
    ax.set(xlabel=variable, ylabel="Probabilidad condicional", title=f"Pago según {variable}", ylim=(0, 1))
    ax.legend(loc="upper right")
    fig.tight_layout()
    if vista_widget:
        return imagen_widget(fig, f"Probabilidad condicional de pago según {variable}")
    plt.show()

In [ ]:
densidad_pago("age", suavizado=1.0)  # Cambia la variable o prueba, por ejemplo, 0.5 y 2.0.

#### Microejercicio práctico — modificar el suavizado

Usa otra variable numérica de `var_num` y prueba un suavizado mayor que el del ejemplo. Completa los dos argumentos de `densidad_pago`; compara la forma de las curvas con la anterior.

In [ ]:
try:
    densidad_pago(..., suavizado=...)  # Completa con una variable de var_num distinta de age y un suavizado, por ejemplo 2.0.
except Exception as error:
    plt.close("all")  # Cierra una figura parcial si los argumentos no son válidos.
    print("El microejercicio no está bien implementado todavía. Revisa la variable y el suavizado y vuelve a intentarlo.")
    print(f"Detalle: {type(error).__name__}: {error}")

**Exploración opcional:** selecciona otra variable numérica y modifica el suavizado sin tocar el ejemplo anterior. Un suavizado muy bajo puede exagerar oscilaciones. Si los controles no aparecen, ejecuta `densidad_pago("age", 2.0)` con los valores que quieras probar.

In [ ]:
selector_densidad = Dropdown(options=var_num, value="age", description="Variable",
                             style={"description_width": "95px"}, layout=Layout(width="300px"))
suavizado_densidad = FloatSlider(value=1.0, min=0.3, max=3.0, step=0.1,
                                 description="Suavizado", continuous_update=False,
                                 style={"description_width": "95px"}, layout=Layout(width="350px"))
visor_densidad = HTML(layout=Layout(width="100%"))

def actualizar_densidad(cambio=None):
    if cambio is not None and cambio["name"] != "value":
        return
    visor_densidad.value = densidad_pago(selector_densidad.value, suavizado_densidad.value,
                                         vista_widget=True)

selector_densidad.observe(actualizar_densidad, names="value")
suavizado_densidad.observe(actualizar_densidad, names="value")
actualizar_densidad()
control_densidad = VBox([
    HTML("<div style='border-left: 4px solid #800000; padding: 8px 12px; "
         "font-size: 18px; font-weight: 600;'>Explora la densidad condicional</div>"),
    HBox([selector_densidad, suavizado_densidad], layout=Layout(align_items="center")),
    visor_densidad,
], layout=Layout(width="100%", gap="12px"))
display(control_densidad)

### Categorías de referencia

Elegimos categorías base para interpretar los coeficientes. Cada coeficiente categórico compara una categoría con su **referencia**, manteniendo constantes los demás predictores. Reordenamos las categorías antes de generar indicadores; con `drop_first=True` la primera se omite. Elegir otra referencia cambiaría los coeficientes individuales, pero **no** las predicciones del mismo modelo.

**Comprueba** en la tabla qué significa, por ejemplo, `foreign=yes` respecto a `foreign=no`. Una referencia puede contener pocos casos: eso afecta a la precisión de la comparación, aunque sea fácil de interpretar.

In [ ]:
referencias = {
    "status": "no checking account",
    "credit.hist": "no credits taken/all credits paid back duly",
    "purpose": "others",
    "savings": "unknown/no savings account",
    "employed.time": "unemployed",
    "status.sex": "male:single",
    "debtors": "none",
    "property": "unknown/no property",
    "installment2": "none",
    "housing": "for free",
    "job": "unskilled - resident",
    "telephone": "none",
    "foreign": "no",
}

for columna, referencia in referencias.items():
    categorias = list(datos[columna].cat.categories)
    datos[columna] = datos[columna].cat.reorder_categories(
        [referencia] + [categoria for categoria in categorias if categoria != referencia]
    )

pd.Series(referencias, name="Categoría de referencia").to_frame()

### Estimar el modelo completo

**¿Por qué no una regresión lineal? (diapositiva 3)** La respuesta solo toma 0 o 1 y no tiene errores normales; además, una recta usada para predecir pagos podría dar valores menores que 0 o mayores que 1. La regresión logística modela el *logit* y transforma el resultado de vuelta al intervalo de probabilidades.

En el modelo binomial, $\log\!\left(p/(1-p)\right)=\eta=\beta_0+\sum_j\beta_j x_j$, donde $p=P(y=\mathrm{yes}\mid x)$. Al invertir la transformación obtenemos $p=e^{\eta}/(1+e^{\eta})=1/(1+e^{-\eta})$. Creamos indicadores para las categorías (`drop_first=True`) y añadimos `const` para el intercepto. `grupos` conserva juntos todos los indicadores de cada variable original para la selección posterior.

Al leer `summary()`, distingue **signo del coeficiente** (dirección del cambio en log-odds), **error estándar** (incertidumbre) y **p-valor** (evidencia bajo el modelo). No interpretes un coeficiente como aumento directo de probabilidad: ese aumento depende del punto de partida y del resto de variables.

#### De la probabilidad a los odds y de vuelta (diapositivas 4–5)

Antes de leer coeficientes, toma un ejemplo **ficticio**: si la probabilidad inicial de pagar es $p=0{,}20$, los *odds* son $p/(1-p)=0{,}20/0{,}80=0{,}25$ y el *logit* es $\log(0{,}25)\approx-1{,}386$. Si un predictor cambia el logit en $\log(2)\approx0{,}693$, los odds se **duplican**: pasan de $0{,}25$ a $0{,}50$. La nueva probabilidad es $0{,}50/(1+0{,}50)=1/3\approx0{,}333$, **no** $0{,}40$.

**Comprueba:** ¿cuánto cambia la probabilidad (en puntos porcentuales)? ¿Por qué duplicar odds no equivale a duplicar probabilidad? En el modelo real, esta comparación mantiene constantes las demás variables; el cambio absoluto de probabilidad depende del perfil inicial. Usa los controles siguientes para comprobarlo con otros valores; si no aparecen, ejecuta `mostrar_html(comparar_odds(0.2, 2))`.

<details><summary>Orientación para docencia</summary>La probabilidad sube de 20 % a 33,3 %: unos 13,3 puntos, no 20. El mismo OR = 2 aplicado a otra probabilidad inicial produce otro incremento absoluto.</details>

In [ ]:
def comparar_odds(probabilidad_inicial, odds_ratio):
    """Ejemplo ficticio: solo varían los odds; no son predicciones del modelo."""
    odds_iniciales = probabilidad_inicial / (1 - probabilidad_inicial)
    odds_finales = odds_iniciales * odds_ratio
    probabilidad_final = odds_finales / (1 + odds_finales)
    return ("<div style='padding: 12px; border-left: 4px solid #800000; font-size: 16px;'>"
            f"Probabilidad inicial: <b>{probabilidad_inicial:.1%}</b> → odds: <b>{odds_iniciales:.3f}</b>"
            f"<br>OR: <b>{odds_ratio:.2f}</b> → odds finales: <b>{odds_finales:.3f}</b>"
            f" → probabilidad final: <b>{probabilidad_final:.1%}</b>"
            f"<br>Cambio: <b>{100 * (probabilidad_final - probabilidad_inicial):+.1f}</b> puntos porcentuales."
            "</div>")

probabilidad_odds = FloatSlider(value=0.2, min=0.05, max=0.95, step=0.05,
                                description="P inicial", continuous_update=False,
                                style={"description_width": "85px"}, layout=Layout(width="350px"))
ratio_odds = FloatSlider(value=2.0, min=0.25, max=5.0, step=0.25,
                         description="OR", continuous_update=False,
                         style={"description_width": "45px"}, layout=Layout(width="350px"))
visor_odds = HTML(layout=Layout(width="100%"))

def actualizar_odds(cambio=None):
    if cambio is not None and cambio["name"] != "value":
        return
    visor_odds.value = comparar_odds(probabilidad_odds.value, ratio_odds.value)

probabilidad_odds.observe(actualizar_odds, names="value")
ratio_odds.observe(actualizar_odds, names="value")
actualizar_odds()
control_odds = VBox([
    HTML("<div style='font-size: 18px; font-weight: 600;'>Probabilidades y odds: ejemplo ficticio</div>"),
    HBox([probabilidad_odds, ratio_odds], layout=Layout(align_items="center")),
    visor_odds,
], layout=Layout(width="100%", gap="12px"))
display(control_odds)

#### Forma de la probabilidad logística (diapositiva 6)

Con una sola variable $x$, la recta $\eta=\beta_0+\beta_1x$ vive en la escala de **log-odds**; la probabilidad $p(x)=1/(1+e^{-\eta})$ tiene forma de S. Las tres curvas siguientes son **didácticas**, no ajustes a los datos: pendiente positiva favorece el pago al crecer $x$, pendiente negativa lo desfavorece, y pendiente nula no cambia la probabilidad. El signo indica dirección, pero el cambio en probabilidad no es constante a lo largo de $x$.

**Comprueba:** ¿qué curva corresponde a $\beta_1=0$? ¿Por qué ninguna curva da probabilidades menores que 0 ni mayores que 1?

In [ ]:
x_ejemplo = np.linspace(-5, 5, 250)
fig, ax = plt.subplots(figsize=(11, 6))
for pendiente, color, etiqueta in [
    (1.0, COLOR_CLAVE, "Efecto positivo"),
    (0.0, COLOR_BORDE, "Sin efecto"),
    (-1.0, COLOR_DATOS, "Efecto negativo"),
]:
    eta = pendiente * x_ejemplo  # Intercepto cero en los tres ejemplos.
    probabilidad = 1 / (1 + np.exp(-eta))
    _ = ax.plot(x_ejemplo, probabilidad, color=color, linewidth=2.5, label=etiqueta)
_ = ax.set(xlabel="Variable x (escala ficticia)", ylabel="Probabilidad de pago",
           title="Efecto de un predictor en la probabilidad", ylim=(0, 1))
_ = ax.legend()
fig.tight_layout()
plt.show()

**Prueba con controles:** mueve el intercepto y la pendiente; la curva usa datos ficticios y **no reajusta** el modelo de crédito. Al cambiar el intercepto se desplaza la probabilidad en $x=0$; al cambiar la pendiente varían dirección e intensidad del efecto. Si no se muestran los controles, ejecuta `mostrar_html(curva_logistica(0, 1))` cambiando los dos argumentos.

In [ ]:
def curva_logistica(intercepto, pendiente):
    rejilla = np.linspace(-5, 5, 250)
    logit = intercepto + pendiente * rejilla
    probas = 1 / (1 + np.exp(-logit))
    fig, ax = plt.subplots(figsize=(11, 6))
    ax.plot(rejilla, probas, color=COLOR_CLAVE, linewidth=2.5, label="Probabilidad")
    ax.axhline(0.5, color=COLOR_BORDE, linestyle="--", alpha=0.65)
    ax.set(xlabel="Variable x (escala ficticia)", ylabel="Probabilidad de pago",
           title="Logit lineal → probabilidad logística", ylim=(0, 1))
    ax.legend()
    fig.tight_layout()
    p_en_cero = 1 / (1 + np.exp(-intercepto))
    return (imagen_widget(fig, "Curva logística didáctica")
            + f"<p>En x = 0: p = <b>{p_en_cero:.1%}</b>. "
            + f"Por cada unidad adicional de x los odds se multiplican por <b>{np.exp(pendiente):.2f}</b> "
            + "(no es un cambio fijo de probabilidad).</p>")

intercepto_curva = FloatSlider(value=0, min=-3, max=3, step=0.25,
                               description="Intercepto", continuous_update=False,
                               style={"description_width": "90px"}, layout=Layout(width="350px"))
pendiente_curva = FloatSlider(value=1, min=-2, max=2, step=0.25,
                              description="Pendiente", continuous_update=False,
                              style={"description_width": "90px"}, layout=Layout(width="350px"))
visor_curva = HTML(layout=Layout(width="100%"))

def actualizar_curva(cambio=None):
    if cambio is not None and cambio["name"] != "value":
        return
    visor_curva.value = curva_logistica(intercepto_curva.value, pendiente_curva.value)

intercepto_curva.observe(actualizar_curva, names="value")
pendiente_curva.observe(actualizar_curva, names="value")
actualizar_curva()
control_curva = VBox([
    HTML("<div style='border-left: 4px solid #800000; padding: 8px 12px; "
         "font-size: 18px; font-weight: 600;'>Explora la función logística</div>"),
    HBox([intercepto_curva, pendiente_curva], layout=Layout(align_items="center")),
    visor_curva,
], layout=Layout(width="100%", gap="12px"))
display(control_curva)

#### ¿Cómo se estiman los coeficientes? (diapositiva 7)

Suponemos que cada resultado binario $y_i$ sigue una distribución Bernoulli con probabilidad condicional $p_i$ y que las observaciones son independientes según el modelo. **Máxima verosimilitud** busca los coeficientes $\beta$ que hacen más plausibles los pagos e impagos observados, maximizando

$$\ell(\beta)=\sum_i\left[y_i\log(p_i)+(1-y_i)\log(1-p_i)\right].$$

`sm.GLM(..., family=sm.families.Binomial()).fit()` hace esa estimación; sus `params` son estimaciones de los coeficientes, **no** probabilidades. Comprueba en el resumen si ha convergido el ajuste. Si una categoría tiene solo pagos o solo impagos (*separación*), algunos coeficientes pueden no estimarse de forma finita o fiable: revisa las frecuencias antes de interpretar OR extremos.

In [ ]:
y = (datos["y"] == "yes").astype(int)
predictores = [columna for columna in datos.columns if columna != "y"]
X = pd.get_dummies(datos[predictores], drop_first=True, prefix_sep="=", dtype=float)
X = sm.add_constant(X, has_constant="add")

grupos = {
    variable: [columna for columna in X if columna == variable or columna.startswith(variable + "=")]
    for variable in predictores
}

In [ ]:
def ajustar_modelo(variables):
    columnas = ["const"] + [columna for variable in variables for columna in grupos[variable]]
    with threadpool_limits(limits=1):
        return sm.GLM(y, X[columnas], family=sm.families.Binomial(), missing="raise").fit(maxiter=100)

In [ ]:
mod_glm0 = ajustar_modelo(predictores)
mod_glm0.summary()

In [ ]:
# La serie params muestra los coeficientes que el modelo ha estimado.
mod_glm0.params

#### Microejercicio práctico — columnas y parámetros

En la celda anterior puedes ver la serie de coeficientes de `mod_glm0`. Usa `len()` —ya aparece en el código anterior— para contar esos coeficientes y los predictores originales de `predictores`. ¿Por qué el primer número puede ser mayor?

In [ ]:
try:
    n_coeficientes = len(...)  # Completa con el atributo de mod_glm0 que contiene los coeficientes estimados.
    n_variables_originales = len(...)  # Completa con la lista de predictores del conjunto original.
    print(f"Predictores originales: {n_variables_originales}; coeficientes (incluido intercepto): {n_coeficientes}")
except Exception as error:
    print("El microejercicio no está bien implementado todavía. Revisa los objetos y vuelve a intentarlo.")
    print(f"Detalle: {type(error).__name__}: {error}")

### Selección automática por AIC

#### Valores ausentes

Todos los modelos candidatos deben usar las mismas observaciones. Comprobamos que no haya valores ausentes.

In [ ]:
datos.isna().sum()

#### Selección hacia atrás por AIC (diapositiva 11)

Partimos del modelo completo. En cada paso ajustamos modelos que retiran **una variable original** (si es categórica, todos sus indicadores), elegimos la retirada que deja el menor AIC y paramos si ninguna lo reduce. El AIC penaliza la complejidad: valores **más bajos** favorecen un compromiso entre ajuste y número de parámetros; no indican calidad predictiva en test.

**Lee** `pasos_aic` de arriba abajo: identifica qué se eliminó primero y cuánto cambió el AIC. Después comprueba qué variables conserva el modelo. Como advierte la presentación, seleccionar automáticamente puede dar resultados inestables y aparentar una capacidad predictiva mayor de la real. Esta búsqueda es local: no garantiza el mejor subconjunto global ni convierte los p-valores posteriores en pruebas independientes de la selección.

In [ ]:
def seleccion_atras(variables):
    if datos.isna().any().any():
        raise ValueError("Resuelve los valores ausentes antes de seleccionar variables.")
    activas = list(variables)
    modelo = ajustar_modelo(activas)
    historial = [{"Paso": 0, "Eliminada": "—", "AIC": modelo.aic, "Variables": len(activas)}]
    while activas:
        candidatos = []
        for variable in activas:
            restantes = [nombre for nombre in activas if nombre != variable]
            candidato = ajustar_modelo(restantes)
            candidatos.append((candidato.aic, variable, candidato))
        aic, retirada, candidato = min(candidatos, key=lambda resultado: resultado[0])
        if aic >= modelo.aic - 1e-7:
            break
        activas.remove(retirada)
        modelo = candidato
        historial.append({"Paso": len(historial), "Eliminada": retirada,
                          "AIC": aic, "Variables": len(activas)})
    return modelo, activas, pd.DataFrame(historial)

In [ ]:
mod_glm1, variables_seleccionadas, pasos_aic = seleccion_atras(predictores)
pasos_aic

In [ ]:
variables_seleccionadas
mod_glm1.summary()

#### Microejercicio práctico — medir el efecto de la selección

Compara el AIC del modelo completo con el seleccionado y calcula cuántas variables originales se retiraron. Reutiliza `mod_glm0`, `mod_glm1`, `predictores` y `variables_seleccionadas`.

In [ ]:
try:
    reduccion_aic = ...  # Resta el AIC de mod_glm1 al AIC de mod_glm0.
    variables_retiradas = ...  # Compara la cantidad de predictores iniciales y seleccionados.
    print(f"Reducción de AIC: {reduccion_aic:.2f}")
    print(f"Variables retiradas: {variables_retiradas}")
except Exception as error:
    plt.close("all")
    print("El microejercicio no está bien implementado todavía. Revisa las operaciones y vuelve a intentarlo.")
    print(f"Detalle: {type(error).__name__}: {error}")

#### Pausa de interpretación · Selección

**Pregunta:** ¿qué variable se retiró primero y cuántas variables originales quedan al final? Explica por qué un AIC menor no garantiza un mejor resultado en test. Usa la tabla de pasos, no solo los p-valores de los coeficientes.

<details><summary>Orientación para docencia</summary>En estos datos se retira primero `property`; el modelo pasa de 20 variables originales a 11. La selección usa solo el ajuste penalizado sobre entrenamiento; la generalización se comprobará después en test.</details>

#### Importancia según la significación estadística

Ordenamos los coeficientes por el valor absoluto de su estadístico z, excluyendo el intercepto.
Este criterio describe evidencia estadística; no mide por sí solo el tamaño del efecto.

In [ ]:
importancia = pd.DataFrame({
    "Importancia (|z|)": mod_glm1.tvalues.drop("const").abs(),
    "p-valor": mod_glm1.pvalues.drop("const"),
}).sort_values("Importancia (|z|)", ascending=False)
importancia

### Validación en entrenamiento

#### Pagos esperados y observados por grupos de probabilidad

Ordenamos las probabilidades predichas y dividimos los créditos en grupos de tamaño parecido mediante cuantiles. En cada grupo comparamos **pagos observados** (suma de unos) y **esperados** (suma de probabilidades). La suma esperada puede no ser entera. El gráfico usa 10 grupos: cambia el argumento de `pagos_por_intervalo()` y vuelve a ejecutar la celda para comprobar si el diagnóstico depende de la agrupación.

Estas son comprobaciones **dentro de entrenamiento**, por lo que pueden ser optimistas. Si hay empates, `qcut` puede formar menos grupos de los solicitados; la tabla muestra el número efectivo y también incluye el mínimo predicho. La comparación entre grupos no sustituye al gráfico de calibración posterior.

In [ ]:
pr = mod_glm1.predict(X[mod_glm1.model.exog_names])

def tabla_por_cuantiles(observado, predicho, grupos=10):
    tabla = pd.DataFrame({"observado": np.asarray(observado), "predicho": np.asarray(predicho)})
    tabla["intervalo"] = pd.qcut(tabla["predicho"], q=grupos, duplicates="drop")
    resumen = tabla.groupby("intervalo", observed=True).agg(
        n=("observado", "size"), obs=("observado", "sum"), exp=("predicho", "sum"),
    )
    return resumen

validacion = tabla_por_cuantiles(y, pr)
validacion

In [ ]:
def pagos_por_intervalo(grupos):
    tabla = tabla_por_cuantiles(y, pr, grupos)
    posiciones = np.arange(1, len(tabla) + 1)
    fig, ax = plt.subplots()
    ancho = 0.36
    ax.bar(posiciones - ancho / 2, tabla["obs"], width=ancho, color=COLOR_DATOS,
           edgecolor=COLOR_BORDE, label="Pagos observados")
    ax.bar(posiciones + ancho / 2, tabla["exp"], width=ancho, color=COLOR_CLAVE,
           edgecolor=COLOR_BORDE, label="Pagos esperados")
    ax.set(xlabel="Grupos de probabilidad (de menor a mayor)", ylabel="Número de pagos",
           title="Pagos observados y esperados", xticks=posiciones)
    ax.legend()
    fig.tight_layout()
    plt.show()

In [ ]:
pagos_por_intervalo(10)  # Cambia el número de grupos y vuelve a ejecutar.

#### Test de Hosmer–Lemeshow (diapositiva 15)

Compara pagos y no pagos observados con sus frecuencias esperadas en grupos de probabilidad. En cada grupo, los **pagos esperados** son la suma de probabilidades predichas y los **observados** son el número de pagos; se compara también el complemento (impagos). Un **p-valor pequeño** (por ejemplo, < 0,05) aporta evidencia de falta de ajuste; uno grande **no demuestra** que el modelo sea correcto. Usamos hasta 10 grupos y grados de libertad iguales al número de grupos efectivamente formados menos 2.

**Diferencia con la presentación:** la diapositiva ilustra intervalos de igual **amplitud** (0–0,1; 0,1–0,2; …); aquí `qcut` usa **cuantiles**, con tamaños aproximadamente iguales, y el gráfico de calibración posterior usa intervalos de igual amplitud. Los cortes pueden cambiar el estadístico y el p-valor. La tabla siguiente cuenta cuántos créditos habría en los intervalos de la diapositiva; compárala con los tamaños `n` de los cuantiles. Ninguna agrupación sustituye a examinar la calibración: con muchos datos el test puede detectar discrepancias muy pequeñas y con pocos casos por grupo su aproximación puede ser poco fiable.

In [ ]:
def hosmer_lemeshow(observado, predicho, grupos=10):
    tabla = tabla_por_cuantiles(observado, predicho, grupos)
    if len(tabla) <= 2:
        raise ValueError("Se necesitan más de dos grupos de probabilidad distintos.")
    no_obs = tabla["n"] - tabla["obs"]
    no_exp = tabla["n"] - tabla["exp"]
    if (tabla["exp"] <= 0).any() or (no_exp <= 0).any():
        raise ValueError("Hay grupos con frecuencia esperada cero.")
    estadistico = (((tabla["obs"] - tabla["exp"]) ** 2 / tabla["exp"])
                   + ((no_obs - no_exp) ** 2 / no_exp)).sum()
    gl = len(tabla) - 2
    return pd.Series({"Chi-cuadrado": estadistico, "Grados de libertad": gl,
                      "p-valor": chi2.sf(estadistico, gl)})

In [ ]:
hosmer_lemeshow(y, pr)

# Comparación de tamaños: cada fila representa el orden del grupo, NO el mismo intervalo de probabilidad.
pd.DataFrame({"n (cuantiles)": validacion["n"].reset_index(drop=True).reindex(range(10)).to_numpy(),
              "n (anchura 0,1)": pd.cut(pr, bins=np.linspace(0, 1, 11),
                                        include_lowest=True).value_counts().sort_index().to_numpy()},
             index=pd.Index(range(1, 11), name="Grupo (de menor a mayor)"))

#### Microejercicio práctico — sensibilidad del test de Hosmer–Lemeshow

Repite el contraste con cinco grupos en lugar de diez. Compara el estadístico y el p-valor con la salida anterior: la agrupación puede afectar al resultado, así que no lo interpretes como una prueba definitiva de calibración.

In [ ]:
try:
    resultado_hl_5 = hosmer_lemeshow(y, pr, grupos=...)  # Completa con cinco grupos.
    display(resultado_hl_5)
except Exception as error:
    print("El microejercicio no está bien implementado todavía. Revisa el número de grupos y vuelve a intentarlo.")
    print(f"Detalle: {type(error).__name__}: {error}")

### Interpretación mediante odds ratios (diapositivas 8–10)

Los *odds* de pago son $p/(1-p)$; un *odds ratio* (OR) compara esos odds entre dos perfiles que solo difieren en el predictor estudiado. Exponenciamos el coeficiente: $\mathrm{OR}=e^{\beta}$. Un OR > 1 multiplica al alza los odds de **pagar** y un OR < 1 los reduce; no es una multiplicación de la probabilidad ni describe causalidad.

**Ejemplo 2 × 2 de la diapositiva 8:** 40 ventas y 20 no ventas vía web dan odds de venta $40/20=2$; 20 ventas y 40 no ventas por correo dan odds $20/40=0{,}5$. El OR web/correo es $2/0{,}5=4$. Sin embargo, las probabilidades son $40/60\approx0{,}67$ y $20/60\approx0{,}33$: la probabilidad **no** se cuadruplica. En la tabla el OR es una comparación sin ajustar; el OR de una regresión compara perfiles con los demás predictores constantes.

**Nota al comparar con la diapositiva 10:** allí se usa otra **categoría de referencia** para `status` y se describen como «probabilidades» cambios que corresponden a **odds**. Por ello pueden cambiar el signo y el valor de los coeficientes impresos sin cambiar las probabilidades que predice el mismo modelo. Usa las referencias explícitas de este cuaderno para interpretar sus coeficientes.

#### Variable categórica

`foreign=yes` se compara con la referencia `foreign=no`, manteniendo constantes los demás predictores. **Ejercicio:** interpreta el resultado en una frase que mencione qué es el evento, la referencia y la diferencia entre odds y probabilidad.

In [ ]:
try:
    odds_ratio_extranjero = np.exp(mod_glm1.params["foreign=yes"])
    print(f"Odds ratio de foreign=yes frente a foreign=no: {odds_ratio_extranjero:.3f}")
except Exception as error:
    print("El ejercicio no está bien implementado todavía. Revisa el modelo y vuelve a intentarlo.")
    print(f"Detalle: {type(error).__name__}: {error}")

#### Variable numérica: edad

El OR de `age` compara perfiles que difieren en **un año** de edad. Para una diferencia de $k$ años, el multiplicador de los odds sería $e^{k\beta_{\mathrm{age}}}$, si se mantiene la relación lineal de edad con el logit. **Pregunta:** ¿es razonable aplicar sin más esa extrapolación a edades fuera del rango observado?

In [ ]:
try:
    odds_ratio_edad = np.exp(mod_glm1.params["age"])
    print(f"Odds ratio por un año adicional de edad: {odds_ratio_edad:.3f}")
except Exception as error:
    print("El ejercicio no está bien implementado todavía. Revisa el modelo y vuelve a intentarlo.")
    print(f"Detalle: {type(error).__name__}: {error}")

#### Microejercicio práctico — comparar diez años de diferencia

En la celda anterior obtuviste el OR asociado a un año. Usa el mismo coeficiente y la propiedad exponencial para calcular el multiplicador de los *odds* entre dos perfiles separados por diez años. No lo interpretes como un cambio fijo de probabilidad.

In [ ]:
try:
    odds_ratio_edad_10 = np.exp(... * mod_glm1.params["age"])  # Reutiliza la expresión del OR de un año; completa el multiplicador.
    print(f"OR para diez años adicionales: {odds_ratio_edad_10:.3f}")
except Exception as error:
    print("El microejercicio no está bien implementado todavía. Revisa el cálculo y vuelve a intentarlo.")
    print(f"Detalle: {type(error).__name__}: {error}")

#### Pausa de interpretación · Efectos

**Pregunta:** compara el OR de `foreign=yes` con la categoría `foreign=no` y el OR de un año adicional de `age`. ¿Cuál es el evento modelado? ¿Puedes traducir el OR de edad a un aumento fijo de probabilidad para todas las personas?

<details><summary>Orientación para docencia</summary>El evento es pagar (`y=yes`). Un OR menor que 1 para `foreign=yes` indica menores odds de pago que para la referencia, manteniendo las demás variables; el de edad compara dos perfiles separados por un año. Ningún OR indica directamente un incremento constante de probabilidad. No se debe presentar esta asociación como causa ni justificar con ella una política crediticia.</details>

#### Intervalos de confianza para los odds ratios

Construimos intervalos del 95 % mediante **perfil de verosimilitud**: fijamos un coeficiente, volvemos a estimar el resto y buscamos los límites que cumplen el contraste de razón de verosimilitudes. Exponenciar los límites transforma el intervalo de log-odds en un intervalo del OR. Esta operación ajusta muchos modelos y puede tardar unos segundos.

**Comprueba** si el intervalo incluye 1: es el valor de «sin cambio en odds». Un intervalo amplio advierte de incertidumbre; tras seleccionar variables con los mismos datos, la cobertura nominal del 95 % puede no mantenerse exactamente.

In [ ]:
def intervalos_perfil(modelo, nivel=0.95):
    matriz = np.asarray(modelo.model.exog, dtype=float)
    respuesta = np.asarray(modelo.model.endog, dtype=float)
    beta = np.asarray(modelo.params)
    errores = np.asarray(modelo.bse)
    corte = chi2.ppf(nivel, df=1)
    limites = []
    with threadpool_limits(limits=1):
        for j, centro in enumerate(beta):
            otras = np.arange(len(beta)) != j

            def diferencia(valor):
                ajuste = sm.GLM(respuesta, matriz[:, otras], offset=matriz[:, j] * valor,
                                family=sm.families.Binomial()).fit(maxiter=100, tol=1e-9)
                if not ajuste.converged or not np.isfinite(ajuste.llf):
                    raise RuntimeError(f"El perfil no converge para {modelo.params.index[j]}.")
                return 2 * (modelo.llf - ajuste.llf) - corte

            extremos = []
            for signo in [-1, 1]:
                paso = max(2 * errores[j], 1e-8)
                extremo = centro + signo * paso
                for _ in range(30):
                    if diferencia(extremo) >= 0:
                        break
                    paso *= 2
                    extremo = centro + signo * paso
                else:
                    raise RuntimeError("No se encontró un límite finito para el perfil.")
                inferior, superior = sorted([centro, extremo])
                extremos.append(brentq(diferencia, inferior, superior, xtol=1e-10))
            limites.append(extremos)
    return pd.DataFrame(limites, index=modelo.params.index, columns=["2.5 %", "97.5 %"])

In [ ]:
IC = intervalos_perfil(mod_glm1)
np.exp(IC).round(2)

### Estimación de la probabilidad de pago

#### Probabilidades predichas

Cada entrada de `pr` está entre 0 y 1 y se refiere a $P(y=\mathrm{yes}\mid x)$ en **entrenamiento**. La probabilidad describe incertidumbre, no una etiqueta correcta o incorrecta hasta que elijamos un umbral.

A continuación buscamos los extremos y contrastamos su respuesta observada: una persona con probabilidad alta puede no pagar y viceversa. Los valores extremos también deben interpretarse con cautela si corresponden a combinaciones de características poco frecuentes.

In [ ]:
pr = mod_glm1.predict(X[mod_glm1.model.exog_names])
pr

#### Observación con la mayor probabilidad de pago

In [ ]:
pos_max = pr.idxmax()
pr.loc[pos_max]
datos.loc[pos_max, "y"]

#### Observación con la menor probabilidad de pago

In [ ]:
pos_min = pr.idxmin()
pr.loc[pos_min]
datos.loc[pos_min, "y"]

#### Distribución de probabilidades según el pago observado

Comparamos las probabilidades predichas de quienes **no pagaron** y de quienes **pagaron**. En cada caja, la línea central es la mediana y la caja cubre el 50 % central. Si las distribuciones se solapan, ningún umbral separará perfectamente ambos grupos; una diferencia entre medianas por sí sola no demuestra calibración.

**Compara** este gráfico de entrenamiento con el correspondiente de test: ¿se mantiene la separación?

In [ ]:
def cajas_probabilidad(probabilidades, respuesta, titulo):
    fig, ax = plt.subplots(figsize=(11, 6))
    grupos = [np.asarray(probabilidades)[np.asarray(respuesta) == clase] for clase in ["no", "yes"]]
    cajas = ax.boxplot(grupos, patch_artist=True, medianprops={"color": COLOR_CLAVE, "linewidth": 2},
                       boxprops={"edgecolor": COLOR_BORDE}, whiskerprops={"color": COLOR_BORDE},
                       capprops={"color": COLOR_BORDE}, flierprops={"markeredgecolor": COLOR_BORDE})
    for caja in cajas["boxes"]:
        caja.set_facecolor(COLOR_DATOS)
    ax.set_xticks([1, 2])
    ax.set_xticklabels(["No paga", "Paga"])
    ax.set(xlabel="Pago observado", ylabel="Probabilidad predicha de pago",
           ylim=(0, 1), title=f"Probabilidades de pago — {titulo}")
    fig.tight_layout()
    plt.show()

cajas_probabilidad(pr, datos["y"], "entrenamiento")

### Curva ROC y AUC (diapositivas 12–14)

Al variar el umbral, la curva ROC representa la **sensibilidad** (pagos correctamente identificados entre todos los pagos) frente a $1-\text{especificidad}$ (impagos clasificados como pagos entre todos los impagos). Cada corte corresponde a un punto $(1-\text{especificidad},\ \text{sensibilidad})$; el ejemplo de la diapositiva 13 da $(0{,}50;0{,}75)$ con el corte 0,4. La diagonal representa clasificación aleatoria.

El **AUC** es el área bajo esa curva: también puede interpretarse como la probabilidad de que, al elegir al azar una persona que pagó y otra que no, el modelo asigne mayor puntuación a quien pagó (en empates, medio acierto). Valores de 0,5 y 1 indican discriminación aleatoria y perfecta, respectivamente. Los rangos «mala/regular/buena» de la diapositiva 14 son **orientativos**, no umbrales universales de calidad.

**Atención:** ROC/AUC miden discriminación, **no calibración** ni costes de los errores. Compara el AUC de entrenamiento con el de test antes de sacar conclusiones sobre generalización.

In [ ]:
def dibujar_roc(observado, probabilidades, titulo):
    fpr, tpr, _ = roc_curve(observado, probabilidades)
    area = roc_auc_score(observado, probabilidades)
    fig, ax = plt.subplots(figsize=(8, 7))
    ax.plot(fpr, tpr, color=COLOR_CLAVE, linewidth=2.5, label=f"Modelo (AUC = {area:.3f})")
    ax.plot([0, 1], [0, 1], "--", color=COLOR_BORDE, label="Azar")
    ax.set(xlabel="1 − Especificidad", ylabel="Sensibilidad", title=titulo,
           xlim=(0, 1), ylim=(0, 1))
    ax.legend(loc="lower right")
    fig.tight_layout()
    plt.show()

dibujar_roc(y, pr, "Curva ROC — entrenamiento")

In [ ]:
roc_auc_score(y, pr)

### Calibración del modelo

Agrupamos las predicciones en intervalos de **igual amplitud** (a diferencia de los cuantiles usados en Hosmer–Lemeshow). Cada punto compara la probabilidad media predicha (eje horizontal) con la proporción real de pagos (vertical). Sobre la diagonal hay concordancia perfecta; por debajo el modelo **sobreestima** los pagos y por encima los **subestima**.

Primero vemos el gráfico de entrenamiento; más adelante repetimos el mismo cálculo en **test** sin reajustar el modelo. La tabla indica el tamaño `n` de cada grupo: un punto basado en pocas personas es muy incierto. **Pregunta:** ¿un AUC alto aseguraría que estos puntos estén sobre la diagonal?

<details><summary>Orientación para docencia</summary>No. AUC evalúa la capacidad de ordenar los pagos por encima de los impagos; calibración compara frecuencias reales con probabilidades predichas. Los intervalos vacíos no producen puntos.</details>

In [ ]:
def calibracion_modelo(observado, probabilidades, intervalos=10, titulo="entrenamiento"):
    tabla = pd.DataFrame({"observado": np.asarray(observado),
                          "predicho": np.asarray(probabilidades)})
    tabla["intervalo"] = pd.cut(tabla["predicho"], bins=np.linspace(0, 1, intervalos + 1),
                                 include_lowest=True)
    resumen = tabla.groupby("intervalo", observed=True).agg(
        n=("observado", "size"), predicho=("predicho", "mean"), observado=("observado", "mean"),
    )
    fig, ax = plt.subplots(figsize=(8, 7))
    ax.plot(resumen["predicho"], resumen["observado"], "o-", color=COLOR_CLAVE, linewidth=2,
            markerfacecolor=COLOR_DATOS, markeredgecolor=COLOR_BORDE, markersize=9,
            label="Modelo")
    ax.plot([0, 1], [0, 1], "--", color=COLOR_BORDE, label="Calibración perfecta")
    ax.set(xlabel="Probabilidad predicha", ylabel="Proporción observada de pagos",
           title=f"Calibración — {titulo}", xlim=(0, 1), ylim=(0, 1))
    ax.legend()
    fig.tight_layout()
    plt.show()
    display(resumen.rename(columns={"predicho": "Probabilidad media", "observado": "Proporción de pagos"}).round(3))

In [ ]:
calibracion_modelo(y, pr, intervalos=10, titulo="entrenamiento")

#### Microejercicio práctico — cambiar los intervalos de calibración

Repite la gráfica con menos intervalos que el ejemplo. Antes de mirar el resultado, predice qué podría ocurrir con el tamaño de cada grupo y con la variabilidad de sus proporciones observadas.

In [ ]:
try:
    calibracion_modelo(y, pr, intervalos=..., titulo="entrenamiento")  # Completa un número menor que los diez grupos del ejemplo.
except Exception as error:
    plt.close("all")  # Cierra una figura parcial si el número de intervalos no es válido.
    print("El microejercicio no está bien implementado todavía. Revisa el número de intervalos y vuelve a intentarlo.")
    print(f"Detalle: {type(error).__name__}: {error}")

### Fijar la regla de decisión **antes** de mirar test

La probabilidad no es una decisión. Para la demostración **fijamos de antemano** $c=0{,}5$: clasificamos como «pagará» y concedemos el crédito si $p>c$. Con probabilidades bien calibradas y costes **iguales** para ambos tipos de error, 0,5 sería una regla coherente; en préstamos reales, costes, equidad y calibración deben estudiarse antes de escoger el umbral. **No elegiremos el mejor corte mirando el test.**

La tabla siguiente compara 0,3, 0,5 y 0,7 **solo en entrenamiento**, a modo de ejemplo del compromiso sensibilidad–especificidad. Estas métricas son optimistas porque el modelo ya vio esos datos: no las uses para proclamar que uno de esos cortes es óptimo. Para elegir un umbral empíricamente haría falta separar una validación interna (o usar predicciones fuera de pliegue) **antes de ajustar y seleccionar variables**, y reservar test para una única evaluación final.

**Pregunta:** ¿cuál de los dos errores aumentaría al elevar el corte? ¿Puede justificarse conceder automáticamente con solo este modelo?

In [ ]:
def metricas_corte(observado, probabilidades, corte):
    real = np.asarray(observado, dtype=bool)
    predicho = np.asarray(probabilidades) > corte
    return {"Umbral": corte,
            "Sensibilidad": (predicho & real).sum() / real.sum(),
            "Especificidad": (~predicho & ~real).sum() / (~real).sum(),
            "Exactitud": (predicho == real).mean()}

# Comparación didáctica en entrenamiento; NO escoger el umbral con el conjunto de test.
pd.DataFrame([metricas_corte(y, pr, corte) for corte in (0.3, 0.5, 0.7)]).round(3)
UMBRAL_FIJADO = 0.5  # Regla preespecificada para la evaluación en test.

**Exploración didáctica en entrenamiento:** mueve el corte para conectar matriz de confusión, sensibilidad y especificidad con un punto de la ROC. El control **no modifica** `UMBRAL_FIJADO`, no selecciona un umbral óptimo y no consulta test. El modelo ya vio estos datos, por lo que las métricas pueden ser optimistas. Si el widget no aparece, ejecuta `mostrar_html(explorar_corte_entrenamiento(0.5))` con otro valor.

In [ ]:
def explorar_corte_entrenamiento(corte):
    """Exploración en entrenamiento; nunca cambia la regla preespecificada para test."""
    metricas = metricas_corte(y, pr, corte)
    decisiones = np.where(np.asarray(pr) > corte, "si", "no")
    resultados = np.where(np.asarray(y) == 1, "yes", "no")
    matriz = pd.crosstab(pd.Series(decisiones), pd.Series(resultados)).reindex(
        index=["no", "si"], columns=["no", "yes"], fill_value=0)
    fpr, tpr, _ = roc_curve(y, pr)
    fig, ax = plt.subplots(figsize=(8, 7))
    ax.plot(fpr, tpr, color=COLOR_CLAVE, linewidth=2.5,
            label=f"Entrenamiento (AUC = {roc_auc_score(y, pr):.3f})")
    ax.plot([0, 1], [0, 1], "--", color=COLOR_BORDE, label="Azar")
    ax.scatter(1 - metricas["Especificidad"], metricas["Sensibilidad"],
               s=150, color=COLOR_DATOS, edgecolor=COLOR_BORDE, zorder=3,
               label=f"Corte explorado = {corte:.2f}")
    ax.set(xlabel="1 − Especificidad", ylabel="Sensibilidad",
           title="Umbrales en entrenamiento (no en test)", xlim=(0, 1), ylim=(0, 1))
    ax.legend(loc="lower right")
    fig.tight_layout()
    return (imagen_widget(fig, "Curva ROC de entrenamiento con el corte explorado")
            + "<p><b>Filas: decisión; columnas: pago observado.</b> Conteos en entrenamiento.</p>"
            + matriz.to_html(border=0)
            + f"<p>Sensibilidad: <b>{metricas['Sensibilidad']:.1%}</b> · "
            + f"Especificidad: <b>{metricas['Especificidad']:.1%}</b> · "
            + f"Exactitud: <b>{metricas['Exactitud']:.1%}</b></p>"
            + f"<p>Regla de test, sin cambios: <b>{UMBRAL_FIJADO:.2f}</b>.</p>")

selector_corte = FloatSlider(value=0.5, min=0.05, max=0.95, step=0.05,
                             description="Corte", continuous_update=False,
                             style={"description_width": "85px"}, layout=Layout(width="450px"))
visor_corte = HTML(layout=Layout(width="100%"))

def actualizar_corte(cambio=None):
    if cambio is not None and cambio["name"] != "value":
        return
    visor_corte.value = explorar_corte_entrenamiento(selector_corte.value)

selector_corte.observe(actualizar_corte, names="value")
actualizar_corte()
control_corte = VBox([
    HTML("<div style='border-left: 4px solid #800000; padding: 8px 12px; "
         "font-size: 18px; font-weight: 600;'>Explora los cortes SOLO en entrenamiento</div>"),
    selector_corte, visor_corte,
], layout=Layout(width="100%", gap="12px"))
display(control_corte)

#### Antes de convertir el modelo en una decisión real

El conjunto de datos contiene `status.sex` (incluye sexo), `foreign` (condición de trabajador extranjero) y `age`. Su presencia en una práctica **no autoriza su uso en una política crediticia**: pueden afectar a grupos de forma desigual y estar sujetos a restricciones legales según el lugar y el contexto. Incluso retirarlos no elimina necesariamente los sesgos, porque otros predictores pueden actuar como sustitutos.

**Debate breve:** ¿qué habría que comprobar antes de usar estas predicciones fuera del aula? Como mínimo: licitud y pertinencia de las variables, calidad y representatividad de los datos, calibración y tasas de error por grupos con muestras suficientes, costes de errores y supervisión humana. Con el tamaño de algunos subgrupos, un diagnóstico por grupo también puede ser muy incierto. **Este ejemplo enseña estadística, no recomienda conceder o denegar préstamos reales.**

<details><summary>Orientación para docencia</summary>La asociación del OR de `foreign` no es causal ni justifica una discriminación. La revisión legal y de equidad debe preceder al despliegue; quitar una variable protegida no demuestra que el modelo sea justo.</details>

### Ejercicio 4.1 — Revisar el modelo con entrenamiento

Trabaja con los resultados de entrenamiento y redacta una conclusión breve, apoyada en una tabla que construyas con código:

1. Compara el AIC y el número de variables originales de `mod_glm0` y `mod_glm1`; revisa en `pasos_aic` qué se retiró primero.
2. Para `foreign=yes`, informa el OR y su intervalo de confianza del 95 % (la tabla `IC` está en escala de coeficientes: exponencia sus límites). Indica la referencia, el evento y si el intervalo incluye OR = 1.
3. Informa el p-valor de Hosmer–Lemeshow en entrenamiento y contrástalo con el gráfico de calibración. Explica por qué un resultado no sustituye al otro ni demuestra cómo funcionará el modelo en test.

Usa solo entrenamiento para esta revisión. No cambies variables ni umbral basándote en los resultados de test.

In [ ]:
try:
    # Construye aquí una tabla que compare AIC y número de predictores de mod_glm0 y mod_glm1.
    # Añade también el primer término retirado según pasos_aic.
    # Calcula el OR y el IC del 95 % para foreign=yes usando mod_glm1 e IC.
    # Resume el p-valor de Hosmer–Lemeshow y contrástalo con la calibración visual.
    # Escribe aquí tu código y muestra los resultados; no consultes ni uses test.
    # Cuando termines, sustituye la línea siguiente por tu solución.
    raise NotImplementedError("Completa el análisis del modelo con los resultados de entrenamiento.")
except NotImplementedError as error:
    print(f"Ejercicio 4.1 pendiente: {error}")
except Exception as error:
    print("El ejercicio no está bien implementado todavía. Revisa tu código y vuelve a intentarlo.")
    print(f"Detalle: {type(error).__name__}: {error}")

## Parte 2. Evaluar los resultados en test

### Leer los datos de test

Este conjunto no intervino en la selección de variables, la estimación **ni la fijación del umbral**. Para predecir aplicamos **las mismas categorías, referencias, nombres y orden de columnas** del entrenamiento; si aparece una categoría nueva, detenemos el proceso en lugar de codificarla silenciosamente. El modelo se evalúa sin volver a ajustar sus coeficientes.

Compara discriminación y calibración fuera de muestra. Después aplica **una vez** la regla `UMBRAL_FIJADO` de la Parte 1. Revisar muchos cortes para elegir el que mejor sale aquí filtraría información del test hacia la decisión: si se hiciera, harían falta nuevos datos para una evaluación final independiente.

In [ ]:
test = pd.read_csv(DATA_PATH / "bank0_test.txt", sep=";")
for columna in datos.select_dtypes(include="category"):
    desconocidas = set(test[columna].dropna()) - set(datos[columna].cat.categories)
    if desconocidas:
        raise ValueError(f"Categorías nuevas en {columna}: {desconocidas}")
    test[columna] = pd.Categorical(test[columna], categories=datos[columna].cat.categories)

X_test = pd.get_dummies(test[predictores], drop_first=True, prefix_sep="=", dtype=float)
X_test = sm.add_constant(X_test, has_constant="add").reindex(columns=X.columns, fill_value=0)
y_test = (test["y"] == "yes").astype(int)

### Predicciones y valores reales

`pr_test` contiene probabilidades de pago para créditos **no utilizados al ajustar el modelo**. Compara su distribución por clase con la de entrenamiento y después contrasta las curvas ROC y los AUC de ambas partes. Una caída en test puede señalar sobreajuste o diferencias entre muestras; un AUC parecido no asegura que las probabilidades estén calibradas.

**Actividad:** identifica qué clase recibe en general probabilidades más altas y si siguen existiendo errores posibles por solapamiento entre cajas.

In [ ]:
pr_test = mod_glm1.predict(X_test[mod_glm1.model.exog_names])
pr_test

In [ ]:
cajas_probabilidad(pr_test, test["y"], "Test")

In [ ]:
dibujar_roc(y_test, pr_test, "Curva ROC — test")

In [ ]:
roc_auc_score(y_test, pr_test)

### Calibración fuera de muestra

Repetimos **sin volver a ajustar** el gráfico y la tabla de calibración en test. Compara los tamaños `n` de los intervalos y cuánto se separan sus puntos de la diagonal respecto a entrenamiento. **Pausa:** ¿un AUC parecido en ambas muestras demostraría que las probabilidades del test son fiables?

<details><summary>Orientación para docencia</summary>No: dos muestras pueden tener AUC similar y calibración distinta. Los puntos alejados de la diagonal y los grupos pequeños requieren cautela; el gráfico tampoco prueba por sí solo que un valor individual sea correcto.</details>

In [ ]:
calibracion_modelo(y_test, pr_test, intervalos=10, titulo="test")

### Sensibilidad, especificidad y exactitud en test

Aplicamos el corte **ya fijado** en entrenamiento: predecimos **pago** cuando $p>\texttt{UMBRAL\_FIJADO}$. En igualdad con el umbral predecimos impago.

- **Sensibilidad**: de quienes pagan, proporción clasificada como pago.
- **Especificidad**: de quienes no pagan, proporción clasificada como impago.
- **Exactitud**: proporción total de clasificaciones correctas.

**Pregunta:** compara estas métricas con la fila del mismo corte en entrenamiento. ¿Por qué maximizar exactitud en test y cambiar ahora la regla impediría considerarlo una evaluación independiente?

In [ ]:
pd.DataFrame({
    "Entrenamiento": metricas_corte(y, pr, UMBRAL_FIJADO),
    "Test": metricas_corte(y_test, pr_test, UMBRAL_FIJADO),
}).round(3)

### Matriz de confusión con la regla fijada

Con el corte `UMBRAL_FIJADO` (0,5 en esta demostración) concedemos crédito si la probabilidad predicha de **pago** lo supera. **Filas**: decisión de conceder (`si`) o no (`no`); **columnas**: resultado real de pago (`yes`) o impago (`no`). La diagonal de decisiones correctas une `no`–`no` y `si`–`yes`.

La primera tabla muestra **conteos**, la segunda **porcentajes dentro de cada fila**: por ejemplo, la celda `si`–`no` dividida por toda la fila `si` es la proporción de impagos entre los créditos concedidos. No la confundas con $1-\text{especificidad}$, cuyo denominador son **todos los impagos reales**. Estos valores describen una evaluación didáctica, no una recomendación de crédito real.

In [ ]:
test["doy_credito"] = np.where(pr_test > UMBRAL_FIJADO, "si", "no")
pd.crosstab(test["doy_credito"], test["y"])

In [ ]:
(100 * pd.crosstab(test["doy_credito"], test["y"], normalize="index")).round(1)

#### Microejercicio práctico — interpretar los créditos concedidos

Reutiliza la expresión de la celda anterior, que calcula porcentajes por fila con `pd.crosstab(..., normalize="index")`. Completa la etiqueta de la fila de créditos concedidos y extrae el porcentaje de `y == "no"`. Ese valor es la proporción de impagos entre los créditos concedidos; fíjate en que su denominador no es el total de impagos.

In [ ]:
try:
    proporciones_test = (100 * pd.crosstab(test["doy_credito"], test["y"], normalize="index")).round(1)
    clase_concedido = ...  # Usa la etiqueta de fila que aparece en la matriz de confusión.
    porcentaje_impago_concedido = proporciones_test.loc[clase_concedido, "no"]  # Extrae la proporción de impago dentro de esa fila.
    print(f"Impagos entre créditos concedidos: {porcentaje_impago_concedido:.1f} %")
except Exception as error:
    plt.close("all")
    print("El microejercicio no está bien implementado todavía. Revisa la etiqueta de fila y vuelve a intentarlo.")
    print(f"Detalle: {type(error).__name__}: {error}")

#### Ubicar la decisión fijada en la curva ROC

El punto burdeos indica la **única regla preespecificada** sobre la ROC de test. Se repiten conteos, porcentajes por fila y métricas con el mismo umbral para conectar cada denominador con el gráfico. La ROC y su AUC resumen **todos los cortes posibles**, pero no autorizan a seleccionar ahora otro corte mirando el resultado del test.

**Actividad:** explica por qué la proporción de impagos entre créditos concedidos no equivale a $1-\text{especificidad}$. ¿Qué cambiarías en el protocolo si el coste de conceder a un impagador fuese mayor que el de rechazar a un pagador? La respuesta es definir costes y validar un nuevo umbral **antes** de usar un conjunto de test independiente.

In [ ]:
def evaluar_umbral_fijado():
    matriz = pd.crosstab(test["doy_credito"], test["y"]).reindex(
        index=["no", "si"], columns=["no", "yes"], fill_value=0,
    )
    porcentajes = matriz.div(matriz.sum(axis=1).replace(0, np.nan), axis=0).mul(100).round(1)
    metricas = metricas_corte(y_test, pr_test, UMBRAL_FIJADO)
    print("Matriz de confusión: filas = decisión; columnas = pago real")
    display(matriz)
    print("Porcentajes por fila (una fila vacía no tiene porcentaje definido)")
    display(porcentajes)
    display(pd.Series(metricas).round(3))
    fpr, tpr, _ = roc_curve(y_test, pr_test)
    fig, ax = plt.subplots(figsize=(8, 7))
    ax.plot(fpr, tpr, color=COLOR_CLAVE, linewidth=2.5,
            label=f"Modelo (AUC = {roc_auc_score(y_test, pr_test):.3f})")
    ax.plot([0, 1], [0, 1], "--", color=COLOR_BORDE, label="Azar")
    ax.scatter(1 - metricas["Especificidad"], metricas["Sensibilidad"], color=COLOR_CLAVE,
               edgecolor=COLOR_BORDE, s=130, zorder=3, label=f"Corte fijado = {UMBRAL_FIJADO:.2f}")
    ax.set(xlabel="1 − Especificidad", ylabel="Sensibilidad", title="Decisión preespecificada — test",
           xlim=(0, 1), ylim=(0, 1))
    ax.legend(loc="lower right")
    fig.tight_layout()
    plt.show()

In [ ]:
evaluar_umbral_fijado()

### Ejercicio 4.2 — Evaluar y comunicar resultados en test

Con el umbral fijado previamente en `UMBRAL_FIJADO`, prepara un breve informe sobre el conjunto de test:

1. Construye la matriz de confusión y calcula sensibilidad, especificidad y exactitud; compara con entrenamiento usando el mismo corte.
2. Calcula el porcentaje de impagos entre los créditos concedidos y deja claro su denominador. Compáralo con la tasa de falsos positivos: no son la misma métrica.
3. Compara AUC y calibración entre entrenamiento y test. Resume una limitación de la evaluación y una precaución de equidad o uso responsable.

No pruebes otros umbrales para decidir cuál funciona mejor en este test: eso lo convertiría en parte del proceso de selección.

In [ ]:
try:
    # Reconstruye la matriz de confusión de test aplicando UMBRAL_FIJADO a pr_test.
    # Calcula sensibilidad, especificidad y exactitud con metricas_corte(...).
    # Compara con entrenamiento usando el mismo umbral y calcula impagos entre concedidos.
    # Llama calibracion_modelo(...) para contrastar la calibración de test con entrenamiento.
    # Muestra una tabla breve y escribe tu conclusión; conserva el umbral fijado.
    # Cuando termines, sustituye la línea siguiente por tu solución.
    raise NotImplementedError("Completa la evaluación y el informe usando el umbral preespecificado.")
except NotImplementedError as error:
    print(f"Ejercicio 4.2 pendiente: {error}")
except Exception as error:
    print("El ejercicio no está bien implementado todavía. Revisa tu código y vuelve a intentarlo.")
    print(f"Detalle: {type(error).__name__}: {error}")

### Cierre de la sesión

Redacta una conclusión breve que responda a estas preguntas usando las tablas y gráficos anteriores:

1. ¿Qué cambió entre el modelo completo y el seleccionado por AIC? ¿Qué variables quedaron?
2. Interpreta un OR categórico y uno numérico sin confundir *odds* con probabilidades.
3. Compara el AUC y la calibración de entrenamiento con los de test. ¿Puede mejorar uno mientras empeora el otro?
4. Comunica sensibilidad, especificidad y proporción de impagos entre los créditos concedidos para el **umbral fijado antes de abrir test**. ¿Qué denominador tiene cada porcentaje? Si quisieras probar otro umbral, ¿qué datos reservarías para evaluar el nuevo protocolo sin sesgo por selección?
5. Identifica al menos una limitación de equidad o legalidad de las variables empleadas; ¿por qué quitar `foreign` no garantizaría automáticamente una decisión justa?

**Límite del análisis:** estas asociaciones no prueban causas; el ejemplo no constituye una política de concesión de crédito. La decisión real exigiría costes explícitos, revisión legal, equidad, supervisión y validación adicional en datos representativos. Al terminar, reinicia el kernel y ejecuta todo en orden para verificar que el informe es reproducible.

### Documentación de las herramientas

- [Modelos lineales generalizados en statsmodels](https://www.statsmodels.org/stable/glm.html).
- [Curva ROC en scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_curve.html).
- [Calibración de probabilidades en scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.calibration.calibration_curve.html).